# Task 6 - Idempotent replay verification

| Stage | Nodes | Edges | Mongo docs | Result |
|---|---:|---:|---:|---|
| Optimum baseline (61 files) | 62,375 | 77,819 | 61 | PASS |
| Add `lab04_replay_probe` to `optimum/version.py` | 62,397 | 77,849 | 61 | PASS |
| Forced unchanged replay | 62,397 | 77,849 | 61 | PASS |
| Restart Spark + forced replay | 62,397 | 77,849 | 61 | PASS |

The modified run replaced a 7-node/6-edge file with 29 nodes and 36 edges and
emitted one stale edge deletion. After restart, Spark reused the same checkpoint:
the stored metadata offset advanced from 126 to 128 without creating another
MongoDB document.

In [1]:
!git -C ../source-repo diff -- optimum/version.py

diff --git a/optimum/version.py b/optimum/version.py
index bb061c5..a41eacc 100644
--- a/optimum/version.py
+++ b/optimum/version.py
@@ -13,3 +13,10 @@
 #  limitations under the License.
 
 __version__ = "2.2.0.dev0"
+
+
+def lab04_replay_probe(value):
+    """Small deterministic branch used only by the Lab 04 replay demo."""
+    if value > 10:
+        return value * 2
+    return value



In [1]:
import subprocess
queries = ['MATCH (n:CPGNode) WITH count(n) AS nodes, count(DISTINCT n.id) AS unique_nodes MATCH ()-[r:CPG_EDGE]->() RETURN nodes, unique_nodes, count(r) AS edges, count(DISTINCT r.id) AS unique_edges']
password = subprocess.run(
    ['docker', 'compose', 'exec', '-T', 'neo4j', 'printenv', 'LAB04_NEO4J_PASSWORD'],
    capture_output=True, text=True, check=True,
).stdout.strip()
for query in queries:
    result = subprocess.run(
        ['docker', 'compose', 'exec', '-T', 'neo4j', 'cypher-shell', '-u', 'neo4j', '-p', password],
        input=query, capture_output=True, text=True, check=True,
    )
    print(result.stdout.rstrip())
# MongoDB verification
!docker compose exec -T mongo mongosh --quiet --eval "db=db.getSiblingDB('lab04'); printjson({documents:db.source_metadata.countDocuments({repo_id:'huggingface/optimum'})}); printjson(db.source_metadata.findOne({_id:'867c06c1f6e594bef9a16137312503a0870399626a16dded522ce7a24143b1a7'}))"

nodes, unique_nodes, edges, unique_edges
62397, 62397, 77849, 77849
{
  documents: 61
}
{
  _id: '867c06c1f6e594bef9a16137312503a0870399626a16dded522ce7a24143b1a7',
  path: 'optimum/version.py',
  content_hash: '079eca803ea5bfe068bc805997b013cc14c9ddc8629a2ec634d12bcebb6720ce',
  processed_at: '2026-07-20T06:02:53.464760Z',
  run_id: '46dff12998f449098b81b50527b2562e',
  kafka_offset: Long('126')
}


In [1]:
!docker compose exec -T spark-metadata bash -lc "for batch in $(find /opt/checkpoints/source-metadata-v1/offsets -maxdepth 1 -type f ! -name '.*' -printf '%f\n' | sort -n | tail -2); do echo batch=$batch; tail -1 /opt/checkpoints/source-metadata-v1/offsets/$batch; done"

batch=8
{"cpg.source-metadata.v1":{"0":126}}batch=9
{"cpg.source-metadata.v1":{"0":128}}


## Reflection

Replay must be demonstrated at the database level, not inferred from producer success. The equality of total/distinct IDs, one Mongo document, changed content hash, and advanced offset together cover the required behavior.